# Manual Clining

In [ ]:
# import pandas as pd
# import json
# import re

# # Load the data
# file_path_processed = '/Users/hasanuddinm/Downloads/dicoding/tfx-practice/data/Political_Bias.csv'
# data = pd.read_csv(file_path_processed)

# # Load the mapping from JSON
# with open('/Users/hasanuddinm/Downloads/dicoding/tfx-practice/mapping.json', 'r') as f:
#     mapping = json.load(f)

# # Convert all column names to lowercase
# data.columns = [col.lower() for col in data.columns]

# # Function to clean text
# def clean_text(text):
#     if isinstance(text, str):
#         # Convert to lowercase
#         text = text.lower()
#         # Remove unusual characters
#         text = re.sub(r'[â€”â€™â—]', '', text)
#     return text

# # Apply the mapping to the bias column
# data['bias'] = data['bias'].map(mapping)

# # Clean the text column
# data['text'] = data['text'].apply(clean_text)

# # Remove rows with any NA values
# data = data.dropna()

# # Select only the 'text' and 'bias' columns
# data = data[['text', 'bias']]

# # Save the processed data to a new CSV file
# output_file_path = '/Users/hasanuddinm/Downloads/dicoding/tfx-practice/data/Political_Bias.csv'
# data.to_csv(output_file_path, index=False)

# print(f"Processed file saved to {output_file_path}")

In [ ]:
# import os
# import csv

# def clean_csv_files(input_dir, output_dir):
#     if not os.path.exists(output_dir):
#         os.makedirs(output_dir)
    
#     for filename in os.listdir(input_dir):
#         if filename.endswith('.csv'):
#             input_path = os.path.join(input_dir, filename)
#             output_path = os.path.join(output_dir, filename)
            
#             with open(input_path, 'rb') as infile, open(output_path, 'w', encoding='utf-8', newline='') as outfile:
#                 reader = csv.reader(infile)
#                 writer = csv.writer(outfile)
                
#                 for row in reader:
#                     cleaned_row = [cell.decode('utf-8', errors='ignore') for cell in row]
#                     writer.writerow(cleaned_row)


In [ ]:
# clean_csv_files("data", "cleaned_data")


# IMPORT LIBRARY

In [ ]:
import tensorflow as tf
from tfx.components import CsvExampleGen, StatisticsGen, SchemaGen, ExampleValidator, Transform, Trainer, Tuner
from tfx.proto import example_gen_pb2
from tfx.orchestration.experimental.interactive.interactive_context import InteractiveContext
import os

# Set Variable

In [ ]:
PIPELINE_NAME = "political-bias-pipeline"
SCHEMA_PIPELINE_NAME = "political-bias-tfdv-schema"

#Directory untuk menyimpan artifact yang akan dihasilkan
PIPELINE_ROOT = os.path.join('yusrilhasan-pipelines', PIPELINE_NAME)

# Path to a SQLite DB file to use as an MLMD storage.
METADATA_PATH = os.path.join('metadata', PIPELINE_NAME, 'metadata.db')

# Output directory where created models from the pipeline will be exported.
SERVING_MODEL_DIR = os.path.join('yusrilhasan-serving_model_dir', PIPELINE_NAME)

# from absl import logging
# logging.set_verbosity(logging.INFO)

In [ ]:
DATA_ROOT = "data"

In [ ]:
interactive_context = InteractiveContext(pipeline_root=PIPELINE_ROOT)

In [ ]:
output = example_gen_pb2.Output(
    split_config = example_gen_pb2.SplitConfig(splits=[
        example_gen_pb2.SplitConfig.Split(name="train", hash_buckets=8),
        example_gen_pb2.SplitConfig.Split(name="eval", hash_buckets=2)
    ])
)
c_input = example_gen_pb2.Input(splits=[
                      example_gen_pb2.Input.Split(name='data', pattern='*.csv')
                     ])
example_gen = CsvExampleGen(input_base=DATA_ROOT, output_config=output, input_config=c_input)

In [ ]:
interactive_context.run(example_gen)

In [ ]:
statistics_gen = StatisticsGen(
    examples=example_gen.outputs["examples"]
)
 
 
interactive_context.run(statistics_gen)

In [ ]:
interactive_context.show(statistics_gen.outputs["statistics"])

In [ ]:
schema_gen = SchemaGen(    statistics=statistics_gen.outputs["statistics"]
)
interactive_context.run(schema_gen)


In [ ]:
interactive_context.show(schema_gen.outputs["schema"])


In [ ]:
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs['statistics'],
    schema=schema_gen.outputs['schema']
)
interactive_context.run(example_validator)


In [ ]:
interactive_context.show(example_validator.outputs['anomalies'])


# Set Transformer

In [ ]:
TRANSFORM_MODULE_FILE = "political_bias_transform.py"


In [ ]:
%%writefile {TRANSFORM_MODULE_FILE}
import tensorflow as tf
LABEL_KEY = "bias"
FEATURE_KEY = "text"
def transformed_name(key):
    """Renaming transformed features"""
    return key + "_xf"
def preprocessing_fn(inputs):
    """
    Preprocess input features into transformed features
    
    Args:
        inputs: map from feature keys to raw features.
    
    Return:
        outputs: map from feature keys to transformed features.    
    """
    
    outputs = {}
    
    outputs[transformed_name(FEATURE_KEY)] = tf.strings.lower(inputs[FEATURE_KEY])
    
    # Ensure the label tensor has the correct shape
    labels = tf.one_hot(inputs[LABEL_KEY], depth=5)
    outputs[transformed_name(LABEL_KEY)] = tf.squeeze(labels, axis=-2)
    # outputs[transformed_name(LABEL_KEY)] = tf.one_hot(inputs[LABEL_KEY], depth=5)
    
    return outputs

In [ ]:
transform  = Transform(
    examples=example_gen.outputs['examples'],
    schema= schema_gen.outputs['schema'],
    module_file=os.path.abspath(TRANSFORM_MODULE_FILE)
)
interactive_context.run(transform)

In [ ]:
def gzip_reader_fn(filenames):
    """Loads compressed data"""
    return tf.data.TFRecordDataset(filenames, compression_type='GZIP')

In [ ]:
TRAINER_MODULE_FILE = "political_bias_trainer.py"

In [ ]:
%%writefile {TRAINER_MODULE_FILE}
import tensorflow as tf
import tensorflow_transform as tft 
from tensorflow.keras import layers
import os  
import tensorflow_hub as hub
from tfx.components.trainer.fn_args_utils import FnArgs
 
LABEL_KEY = "bias"
FEATURE_KEY = "text"
NUM_CLASSES = 5  # Update this to match your number of categories

 
def transformed_name(key):
    """Renaming transformed features"""
    return key + "_xf"
 
def gzip_reader_fn(filenames):
    """Loads compressed data"""
    return tf.data.TFRecordDataset(filenames, compression_type='GZIP')
 
 
def input_fn(file_pattern, 
             tf_transform_output,
             num_epochs,
             batch_size=64)->tf.data.Dataset:
    """Get post_tranform feature & create batches of data"""
    
    # Get post_transform feature spec
    transform_feature_spec = (
        tf_transform_output.transformed_feature_spec().copy())
    
    # create batches of data
    dataset = tf.data.experimental.make_batched_features_dataset(
        file_pattern=file_pattern,
        batch_size=batch_size,
        features=transform_feature_spec,
        reader=gzip_reader_fn,
        num_epochs=num_epochs,
        label_key = transformed_name(LABEL_KEY))
    return dataset
 
# os.environ['TFHUB_CACHE_DIR'] = '/hub_chace'
# embed = hub.KerasLayer("https://tfhub.dev/google/universal-sentence-encoder/4")
 
# Vocabulary size and number of words in a sequence.
VOCAB_SIZE = 5000
SEQUENCE_LENGTH = 100
 
 
embedding_dim=16
def model_builder():
    """Build machine learning model"""
    # Get preprocessed input directly
    inputs = tf.keras.Input(shape=(1,), name=transformed_name(FEATURE_KEY), dtype=tf.string)
    
    # Use embedding with string input
    word_vectors = tf.keras.layers.Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=embedding_dim,
        input_length=SEQUENCE_LENGTH
    )(tf.strings.to_hash_bucket_fast(inputs, VOCAB_SIZE))
    
    x = tf.keras.layers.GlobalAveragePooling1D()(word_vectors)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    
    
    model = tf.keras.Model(inputs=inputs, outputs = outputs)
    
    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(0.01),
        metrics=tf.keras.metrics.CategoricalAccuracy(name='accuracy')
    
    )
    
    # print(model)
    model.summary()
    return model 
 
 
def _get_serve_tf_examples_fn(model, tf_transform_output):
    
    model.tft_layer = tf_transform_output.transform_features_layer()
    
    @tf.function
    def serve_tf_examples_fn(serialized_tf_examples):
        
        feature_spec = tf_transform_output.raw_feature_spec()
        
        feature_spec.pop(LABEL_KEY)
        
        parsed_features = tf.io.parse_example(serialized_tf_examples, feature_spec)
        
        transformed_features = model.tft_layer(parsed_features)
        
        # get predictions using the transformed features
        return model(transformed_features)
        
    return serve_tf_examples_fn
    
def run_fn(fn_args: FnArgs) -> None:
    
    log_dir = os.path.join(os.path.dirname(fn_args.serving_model_dir), 'logs')
    
    tensorboard_callback = tf.keras.callbacks.TensorBoard(
        log_dir = log_dir, update_freq='batch'
    )
    
    es = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', verbose=1, patience=10)
    mc = tf.keras.callbacks.ModelCheckpoint(fn_args.serving_model_dir, monitor='val_accuracy', mode='max', verbose=1, save_best_only=True)
    
    
    # Load the transform output
    tf_transform_output = tft.TFTransformOutput(fn_args.transform_graph_path)
    
    # Create batches of data
    train_set = input_fn(fn_args.train_files, tf_transform_output, 10)
    val_set = input_fn(fn_args.eval_files, tf_transform_output, 10)
    
    # Build the model
    model = model_builder()
    
    
    # Train the model
    model.fit(x = train_set,
            validation_data = val_set,
            callbacks = [tensorboard_callback, es, mc],
            steps_per_epoch = 10, 
            validation_steps= 10,
            epochs=10)
    signatures = {
        'serving_default':
        _get_serve_tf_examples_fn(model, tf_transform_output).get_concrete_function(
                                    tf.TensorSpec(
                                    shape=[None],
                                    dtype=tf.string,
                                    name='examples'))
    }
    model.save(fn_args.serving_model_dir, save_format='tf', signatures=signatures)


In [ ]:
from tfx.proto import trainer_pb2
 
trainer  = Trainer(
    module_file=os.path.abspath("political_bias_trainer.py"),
    examples = transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    train_args=trainer_pb2.TrainArgs(splits=['train']),
    eval_args=trainer_pb2.EvalArgs(splits=['eval'])
)
interactive_context.run(trainer)


In [ ]:
from tfx.dsl.components.common.resolver import Resolver 
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import LatestBlessedModelStrategy 
from tfx.types import Channel 
from tfx.types.standard_artifacts import Model, ModelBlessing 
 
model_resolver = Resolver(
    strategy_class= LatestBlessedModelStrategy,
    model = Channel(type=Model),
    model_blessing = Channel(type=ModelBlessing)
).with_id('Latest_blessed_model_resolver')
 
interactive_context.run(model_resolver)

In [ ]:
import tensorflow_model_analysis as tfma 
 
eval_config = tfma.EvalConfig(
    model_specs=[tfma.ModelSpec(label_key='bias_xf')],
    slicing_specs=[tfma.SlicingSpec()],
    metrics_specs=[
        tfma.MetricsSpec(
            metrics=[
                tfma.MetricConfig(class_name='ExampleCount'),
                tfma.MetricConfig(class_name='Accuracy'),
                tfma.MetricConfig(class_name='AUC')
            ]
        )
    ]
 
)

In [ ]:
from tfx.components import Evaluator
evaluator = Evaluator(
    examples=transform.outputs['transformed_examples'],
    model=trainer.outputs['model'],
    baseline_model=model_resolver.outputs['model'],
    eval_config=eval_config)
 
interactive_context.run(evaluator)

In [ ]:
import tensorflow_model_analysis as tfma

# Path to the evaluation results
eval_result_path = 'yusrilhasan-pipelines/political-bias-pipeline/Evaluator/evaluation/199'

# Load the evaluation results
eval_result = tfma.load_eval_result(eval_result_path)

# Render the slicing metrics
tfma.view.render_slicing_metrics(eval_result)

In [ ]:
from tfx.components import Pusher 
from tfx.proto import pusher_pb2 
 
pusher = Pusher(
model=trainer.outputs['model'],
model_blessing=evaluator.outputs['blessing'],
push_destination=pusher_pb2.PushDestination(
    filesystem=pusher_pb2.PushDestination.Filesystem(
        base_directory='yusrilhasan-serving_model_dir/political-bias-detection-model'))
 
)
 
interactive_context.run(pusher)

In [ ]:
TUNER_MODULE_FILE = "political_bias_tuner.py"

In [ ]:
%%writefile {TUNER_MODULE_FILE}

from tensorflow.keras import layers
import tensorflow as tf
import tensorflow_transform as tft
import os  
import tensorflow_hub as hub
from tfx.components.trainer.fn_args_utils import FnArgs
import keras_tuner as kt

try:
    # Try the most common import paths
    try:
        from tfx.components.tuner.component import TunerFnResult
    except ImportError:
        try:
            from tfx.extensions.google_cloud_ai_platform.tuner.component import TunerFnResult
        except ImportError:
            try:
                from tfx.v1.components.tuner.component import TunerFnResult
            except ImportError:
                # Define our own if none of the imports work
                class TunerFnResult:
                    def __init__(self, tuner, fit_kwargs):
                        self.tuner = tuner
                        self.fit_kwargs = fit_kwargs
except Exception as e:
    print(f"Error importing TunerFnResult: {e}")
    # Define a basic version if all else fails
    class TunerFnResult:
        def __init__(self, tuner, fit_kwargs):
            self.tuner = tuner
            self.fit_kwargs = fit_kwargs
            
LABEL_KEY = "bias"
FEATURE_KEY = "text"
NUM_CLASSES = 5
VOCAB_SIZE = 1000
embedding_dim = 16

def transformed_name(key):
    """Renaming transformed features"""
    return key + "_xf"

def gzip_reader_fn(filenames):
    """Loads compressed data"""
    return tf.data.TFRecordDataset(filenames, compression_type='GZIP')

def input_fn(file_pattern, 
             tf_transform_output,
             num_epochs=1,
             batch_size=64):
    """Get post_transform feature & create batches of data"""
    
    transform_feature_spec = (
        tf_transform_output.transformed_feature_spec().copy())
    
    dataset = tf.data.experimental.make_batched_features_dataset(
        file_pattern=file_pattern,
        batch_size=batch_size,
        features=transform_feature_spec,
        reader=gzip_reader_fn,
        num_epochs=num_epochs,
        label_key=transformed_name(LABEL_KEY))
    return dataset

def model_builder(hp):
    """Build machine learning model with minimal tuning"""
    inputs = tf.keras.Input(shape=(1,), name=transformed_name(FEATURE_KEY), dtype=tf.string)
    
    # Hash the text to integer indices
    hashed_text = tf.strings.to_hash_bucket_fast(inputs, VOCAB_SIZE)
    
    # Reshape to ensure consistent shape
    reshaped_text = tf.reshape(hashed_text, [-1, 1])
    
    # Embedding layer
    word_vectors = tf.keras.layers.Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=embedding_dim
    )(reshaped_text)
    
    x = tf.keras.layers.GlobalAveragePooling1D()(word_vectors)
    
    # Only tune one hyperparameter with explicit default value
    units = hp.Choice('units', values=[32, 64, 128], default=64)
    
    x = tf.keras.layers.Dense(units, activation='relu')(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs)

    # Use a default learning rate to avoid None comparison
    learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4], default=1e-2)
    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate),
        metrics=['accuracy']
    )
    return model

# Critical fix for the specific error
class CustomTuner(kt.RandomSearch):
    """Custom tuner that avoids the None comparison issue"""
    def get_best_models(self, num_models=1):
        """Override to avoid None comparison"""
        if num_models is None:
            num_models = 1
        return super().get_best_models(num_models)

def tuner_fn(fn_args: FnArgs):
    """Build the tuner using the KerasTuner API with fixes for the None comparison issue."""
    
    # Get transform output
    tf_transform_output = tft.TFTransformOutput(fn_args.transform_graph_path)
    
    # Create a custom tuner that handles None values
    tuner = CustomTuner(
        model_builder,
        objective='val_accuracy',
        max_trials=3,
        directory=fn_args.working_dir,
        project_name='political_bias_tuning'
    )
    
    # Create training dataset
    train_dataset = input_fn(
        file_pattern=fn_args.train_files,
        tf_transform_output=tf_transform_output,
        num_epochs=1,
        batch_size=64
    )
    
    # Create validation dataset
    eval_dataset = input_fn(
        file_pattern=fn_args.eval_files,
        tf_transform_output=tf_transform_output,
        num_epochs=1,
        batch_size=64
    )
    
    # Explicitly set non-None values for steps
    train_steps = 100 if fn_args.train_steps is None else fn_args.train_steps
    eval_steps = 50 if fn_args.eval_steps is None else fn_args.eval_steps
    
    return TunerFnResult(
        tuner=tuner,
        fit_kwargs={
            "x": train_dataset,
            "validation_data": eval_dataset,
            "steps_per_epoch": train_steps,  # Use explicit non-None value
            "validation_steps": eval_steps   # Use explicit non-None value
        }
    )


In [ ]:
tuner = Tuner(
        module_file=os.path.join("political_bias_tuner.py"),
        examples=transform.outputs['transformed_examples'],
        transform_graph=transform.outputs['transform_graph'],
        schema=schema_gen.outputs['schema'],
        train_args=trainer_pb2.TrainArgs(splits=['train'], num_steps=500),
        eval_args=trainer_pb2.EvalArgs(splits=['eval'], num_steps=100)
    )

interactive_context.run(tuner, enable_cache=False)  # Disable cache to force fresh execution

In [ ]:
# Add the evaluator to your pipeline components
pipeline_components = [
    example_gen,
    statistics_gen,
    schema_gen,
    example_validator,
    transform,
    trainer,
    evaluator,  # Ensure this is included
    pusher
]

In [ ]:
!pipreqs --force

In [ ]:
from tfx.orchestration import pipeline
from tfx.orchestration.local import local_dag_runner
from tfx.orchestration.metadata import sqlite_metadata_connection_config

pipeline = pipeline.Pipeline(
    pipeline_name='my_pipeline',
    pipeline_root='yusrilhasan-pipelines/political-bias-pipeline',
    components=pipeline_components,
    enable_cache=True,
    metadata_connection_config=sqlite_metadata_connection_config('storage/metadata.db')
)

local_dag_runner.LocalDagRunner().run(pipeline)

In [ ]:
import requests
from pprint import PrettyPrinter
 
pp = PrettyPrinter()
pp.pprint(requests.get("http://localhost:8500/v1/models/political-bias-detection-model").json())

In [ ]:
from prediction import predict

# Define the server URL
SERVER_URL = 'http://localhost:8500/v1/models/political-bias-detection-model:predict'

# Input text
text = "This policy will benefit the economy while protecting our values. Said by the president of the United States. Donald Trump"

predict(SERVER_URL, text)
